# MMTFv3 Backtest — Trial 13 (Transformer)

Standalone backtest notebook with **hardcoded** Trial 13 hyperparameters.
No study or saved `best_params.json` required — just trains the model from scratch
and runs a full OOS backtest.

| Section | Contents |
|---|---|
| Portfolio summary | Sharpe, Sortino, Calmar, profit factor, max-DD |
| Position evolution | Daily avg weight, rolling exposure, per-ticker heatmap |
| Calibration | Position vs realized-return quantile, position-return scatter |
| Trade-level | Avg profit/trade, holding duration, long vs short breakdown |
| Rolling | 30-day rolling Sharpe, rolling drawdown |
| Calendar | Monthly PnL heatmap |
| TC-adjusted | All key metrics repeated net of transaction costs |
| Attribution | Branch weights, sequential feature gates, leakage tests |

**Trial 13**: `val_sharpe=0.1565`, Transformer backbone, d_model=128, 2 layers

## 1. Config & Hardcoded Trial 13 Parameters

In [ ]:
from pathlib import Path
import numpy as np

# ── Must match training notebook ────────────────────────────────────────────
TICKERS                 = ['CL', 'GC']
BACKBONE                = 'transformer'
USE_PTP                 = True
USE_FUSED_SPATIAL       = True
SPATIAL_ENCODER         = 'fused'
SPATIAL_LOOKBACK_BARS   = 8
BAR_MINUTES             = 5
TARGET_HORIZON_MINUTES  = 60
SAMPLE_SESSION          = "overlap"
SAMPLE_SESSION_START    = "06:00"
SAMPLE_SESSION_END      = "13:00"
SAMPLE_STRIDE           = 12
AE_WINDOW               = 21
F_AE                    = 4
USE_AMP                 = True

IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT    = Path('/content/drive/MyDrive/features')
    RESULTS_PATH = Path('/content/drive/MyDrive/results/mmtfv3_trial13')
else:
    DATA_ROOT    = Path('/workspace/')
    RESULTS_PATH = Path('/workspace/results/mmtfv3_trial13')

BACKTEST_PATH = RESULTS_PATH
BACKTEST_PATH.mkdir(parents=True, exist_ok=True)

# ── Backtest params ──────────────────────────────────────────────────────────
BACKTEST_BATCH_SIZE = 256
TRADE_THRESH        = 0.05
TC_COST_BPS         = 0.5
TC_COST             = TC_COST_BPS * 1e-4

BARS_PER_DAY   = int(450 / TARGET_HORIZON_MINUTES)
BARS_PER_YEAR  = 252 * BARS_PER_DAY
ANNUALIZATION  = np.sqrt(BARS_PER_YEAR)

# ── Trial 13 hyperparameters (hardcoded) ─────────────────────────────────────
best = {
    # Architecture
    'backbone':             'transformer',
    'd_model':              128,
    'd_static_emb':         64,
    'n_heads':              4,
    'n_layers':             2,
    'd_ff':                 512,
    'dropout':              0.3453639432602834,
    'grn_dropout':          0.052815089613024946,
    # VAE
    'ae_type':              'vae',
    'd_latent':             32,
    'd_ae_hidden':          64,
    'kl_weight':            0.07897536991484493,
    'recon_weight':         0.21588156182458917,
    # Position state
    'state_hidden_dim':     32,
    'state_momentum':       0.9646586760274916,
    # PTP
    'ptp_temperature':      1.086714191846717,
    'ce_weight':            1.2033416737660658,
    'pnl_weight':           1.027757604686925,
    'profit_scale':         154.48201536718756,
    'inner_threshold':      0.2679012554354161,
    'outer_threshold':      1.3174138929135082,
    'quantile_lr_scale':    0.8644116003149853,
    'ptp_lr_scale':         0.7575199366232556,
    'head_lr_decay':        0.5169105829653088,
    'quantile_patience':    2,
    'ptp_patience':         2,
    'head_overfit_tolerance': 0.049769584746832754,
    # Sequential
    'seq_layers':           3,
    'seq_nheads':           2,
    # Lookbacks
    'tech_lookback':        128,
    'seq_lookback':         48,
    'numbars_lookback':     24,
    # Training
    'batch_size':           96,
    'learning_rate':        0.00016939947065881794,
    'weight_decay':         0.00451828446012055,
    'max_norm':             0.683349754545588,
    # Trading loss
    'tc_cost':              5.2983284294691105e-05,
    'init_direction_weight': 1.4777208599779732,
    'final_direction_weight': 0.14368602318326995,
    'init_reg_weight':      0.25254018901500663,
    'target_exposure':      0.3150225267394001,
    'use_sortino':          True,
    'downside_vol_weight':  0.3373586633475875,
    'holding_weight':       0.1576406187196639,
    # Mamba defaults (unused for transformer but kept for model init)
    'd_state':              16,
    'd_conv':               4,
    'expand':               2,
}

prefix = "trial13_transformer_ptp"
print(f"prefix            : {prefix}")
print(f"RESULTS_PATH      : {RESULTS_PATH}")
print(f"BACKBONE          : {BACKBONE}")
print(f"SPATIAL_ENCODER   : {SPATIAL_ENCODER}")
print(f"BARS_PER_DAY      : {BARS_PER_DAY}   ANNUALIZATION: {ANNUALIZATION:.2f}")
print(f"\nTrial 13 params ({len(best)} keys):")
for k, v in sorted(best.items()):
    print(f"  {k:35s}: {v}")

## 2. Imports

In [ ]:
import json, warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
warnings.filterwarnings("ignore")

from CTAFlow.data.datasets.v3_continuous import (
    V3ContinuousPrep,
    V3ContinuousDataset,
    build_v3_loaders,
    unpack_v3_batch,
    v3_collate_fn,
    loaders_from_samples,
    SessionSpec,
)
from CTAFlow.models.deep_learning.multi_branch.tft import (
    MMTFv3Core,
    StatefulMMTFv3Core,
    returns_to_classes,
    train_epoch_v3_ptp,
    evaluate_v3_ptp,
    train_epoch_v3_stateful,
    evaluate_v3_stateful,
    PTPLoss,
    ContinuousTradingLoss,
    SharpeScheduler,
    HeadAwarePTPScheduler,
    build_ptp_optimizer_param_groups,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 3. Load Data & Build Samples

In [ ]:
print("Loading ticker data...")
prep = V3ContinuousPrep.from_directories(
    root_dir=DATA_ROOT,
    tickers=TICKERS,
    sessions=[SessionSpec("custom", SAMPLE_SESSION_START, SAMPLE_SESSION_END)],
    bar_minutes=BAR_MINUTES,
    target_horizon_minutes=TARGET_HORIZON_MINUTES,
    ae_window=AE_WINDOW,
)

dims = prep.get_dims()
F_TECH               = dims['f_tech']
F_SEQ                = dims['f_seq']
NUMBARS_CHANNELS     = dims['numbars_channels']
VPIN_CHANNELS        = dims.get('vpin_channels', 4)
VPIN_BINS            = dims.get('vpin_bins', 128)
VPIN_TIME            = dims.get('vpin_time', 12)
FUSED_SPATIAL_CHANNELS = dims['fused_spatial_channels']
FUSED_SPATIAL_BINS     = dims['fused_spatial_bins']

print(f"\nFeature dimensions: {dims}")
print(f"Fused spatial shape: ({FUSED_SPATIAL_CHANNELS}, {FUSED_SPATIAL_BINS})")
for ticker in TICKERS:
    nb_ts, nb_vals = prep._numbars_ts.get(ticker, (np.array([], dtype='datetime64[ns]'), np.empty((0, 4, 32), dtype=np.float32)))
    print(f"{ticker}: NumberBars loaded={len(nb_ts):,}, shape={nb_vals.shape}")
    assert len(nb_ts) > 0, f"{ticker} missing NumberBars"

print(f"Tech features ({F_TECH}): {prep._tech_feature_cols[:10]}...")
print(f"n_tickers={prep.n_tickers}, n_asset_classes={prep.n_asset_classes}, n_asset_subclasses={prep.n_asset_subclasses}")

In [ ]:
tech_lookback    = int(best['tech_lookback'])
seq_lookback     = int(best['seq_lookback'])
numbars_lookback = int(best['numbars_lookback'])

print(f"Building samples (tech={tech_lookback}, seq={seq_lookback}, nb={numbars_lookback})...")
_all_samples = prep.build_samples(
    tech_lookback=tech_lookback,
    seq_lookback_bars=seq_lookback,
    numbars_lookback=numbars_lookback,
    use_fused_spatial=USE_FUSED_SPATIAL,
    session_only=True,
    sample_session=SAMPLE_SESSION,
    sample_session_start=SAMPLE_SESSION_START,
    sample_session_end=SAMPLE_SESSION_END,
    stride=SAMPLE_STRIDE,
)
assert _all_samples, "No valid samples produced!"

# Date-based train/val split (80/20)
_all_dates = sorted(set(s["date"] for s in _all_samples))
_n_val = max(1, int(len(_all_dates) * 0.2))
VAL_CUTOFF = _all_dates[-_n_val]

TRAIN_SAMPLES = [s for s in _all_samples if s["date"] < VAL_CUTOFF]
VAL_SAMPLES   = [s for s in _all_samples if s["date"] >= VAL_CUTOFF]
del _all_samples

print(f"Train: {len(TRAIN_SAMPLES):,}  Val: {len(VAL_SAMPLES):,}  (cutoff={VAL_CUTOFF})")
print(f"Total unique days: {len(_all_dates)}")

## 4. Build Model & Train

In [ ]:
# ── Dataloaders ───────────────────────────────────────────────────────────────
train_loader, val_loader = loaders_from_samples(
    TRAIN_SAMPLES, VAL_SAMPLES, prep,
    batch_size=best['batch_size'],
    use_fused_spatial=USE_FUSED_SPATIAL,
    tech_lookback=tech_lookback,
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# ── Model ─────────────────────────────────────────────────────────────────────
base_model = MMTFv3Core(
    f_tech=F_TECH, f_seq=F_SEQ, f_ae=F_AE,
    ae_type=best['ae_type'],
    d_latent=best['d_latent'], d_ae_hidden=best['d_ae_hidden'],
    kl_weight=best['kl_weight'], recon_weight=best['recon_weight'],
    n_tickers=prep.n_tickers,
    n_asset_classes=prep.n_asset_classes,
    n_asset_subclasses=prep.n_asset_subclasses,
    d_model=best['d_model'], d_static_emb=best['d_static_emb'],
    backbone=best['backbone'], n_heads=best['n_heads'], n_layers=best['n_layers'],
    d_ff=best['d_ff'], d_state=best['d_state'],
    d_conv=best['d_conv'], expand=best['expand'],
    dropout=best['dropout'], grn_dropout=best['grn_dropout'],
    spatial_encoder=SPATIAL_ENCODER,
    numbars_channels=NUMBARS_CHANNELS, vpin_channels=VPIN_CHANNELS,
    vpin_bins=VPIN_BINS, vpin_time=VPIN_TIME,
    seq_layers=best['seq_layers'], seq_nheads=best['seq_nheads'],
)

model = StatefulMMTFv3Core(
    base_model=base_model,
    n_tickers=prep.n_tickers,
    quantile_head=USE_PTP,
    ptp_temperature=best['ptp_temperature'],
    state_hidden_dim=best['state_hidden_dim'],
    state_momentum=best['state_momentum'],
    update_on_eval=True,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {best['backbone'].upper()}, d_model={best['d_model']}, params={n_params:,}")

In [ ]:
NUM_EPOCHS = 30
WARMUP_EPOCHS = 5

# ── Loss ──────────────────────────────────────────────────────────────────────
trading_kwargs = dict(
    tc_cost=best['tc_cost'],
    direction_weight=best['init_direction_weight'],
    reg_weight=best['init_reg_weight'],
    target_exposure=best['target_exposure'],
    use_sortino=best['use_sortino'],
    downside_vol_weight=best['downside_vol_weight'],
    tc_in_sharpe=True,
    holding_weight=best['holding_weight'],
)

loss_fn = PTPLoss(
    ce_weight=best['ce_weight'],
    pnl_weight=best['pnl_weight'],
    profit_scale=best['profit_scale'],
    inner_threshold=best['inner_threshold'],
    outer_threshold=best['outer_threshold'],
    **trading_kwargs,
).to(device)

# ── Optimizer ─────────────────────────────────────────────────────────────────
optimizer = optim.AdamW(
    build_ptp_optimizer_param_groups(
        model,
        base_lr=best['learning_rate'],
        weight_decay=best['weight_decay'],
        quantile_lr_scale=best['quantile_lr_scale'],
        ptp_lr_scale=best['ptp_lr_scale'],
    )
)

lr_scheduler = HeadAwarePTPScheduler(
    optimizer=optimizer,
    total_epochs=NUM_EPOCHS,
    head_decay=best['head_lr_decay'],
    quantile_patience=best['quantile_patience'],
    ptp_patience=best['ptp_patience'],
    overfit_tolerance=best['head_overfit_tolerance'],
    ptp_downside_weight=best['downside_vol_weight'],
)

sharpe_sched = SharpeScheduler(
    warmup_epochs=WARMUP_EPOCHS,
    total_epochs=NUM_EPOCHS,
    initial_direction_weight=best['init_direction_weight'],
    final_direction_weight=best['final_direction_weight'],
    initial_reg_weight=best['init_reg_weight'],
    final_reg_weight=0.05,
    initial_target_exposure=0.2,
    final_target_exposure=best['target_exposure'],
    initial_holding_weight=0.0,
    final_holding_weight=best['holding_weight'],
)

_inner_trading_loss = loss_fn.trading_loss
scaler = torch.amp.GradScaler() if (USE_AMP and device.type == 'cuda') else None

print(f"Loss: PTPLoss (ce={best['ce_weight']:.2f}, pnl={best['pnl_weight']:.2f})")
print(f"LR: {best['learning_rate']:.6f}, WD: {best['weight_decay']:.6f}")
print(f"AMP: {'ON' if scaler else 'OFF'}")
print(f"Epochs: {NUM_EPOCHS}, Warmup: {WARMUP_EPOCHS}")

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────────
from CTAFlow.data.datasets.v3_continuous import unpack_v3_batch

best_val_sharpe = -1e9
best_state = None

for epoch in range(1, NUM_EPOCHS + 1):
    sharpe_sched.step(epoch, _inner_trading_loss)

    train_loss, train_metrics = train_epoch_v3_ptp(
        model, train_loader, loss_fn, optimizer, device,
        max_norm=best['max_norm'], scaler=scaler,
        unpack_fn=unpack_v3_batch,
    )

    val_metrics = evaluate_v3_ptp(model, val_loader, loss_fn, device,
                                   unpack_fn=unpack_v3_batch)

    metrics = {**{f"train_{k}": v for k, v in train_metrics.items()},
               **{f"val_{k}": v for k, v in val_metrics.items()}}
    lr_scheduler.step(metrics)

    val_sharpe = val_metrics.get('sharpe', 0) or 0
    tag = ""
    if val_sharpe > best_val_sharpe:
        best_val_sharpe = val_sharpe
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        tag = " *BEST*"

    print(f"  E{epoch:02d} | Loss: {train_loss:.4f} | "
          f"Sharpe: {val_sharpe:.4f} | "
          f"WinRate: {val_metrics.get('win_rate', 0)*100:.1f}% | "
          f"DirAcc: {val_metrics.get('dir_acc', 0)*100:.1f}% | "
          f"Exposure: {val_metrics.get('exposure', 0):.3f} | "
          f"ClsAcc: {val_metrics.get('cls_acc', 0)*100:.1f}%{tag}")

print(f"\nBest val Sharpe: {best_val_sharpe:.4f}")

# Load best state
if best_state:
    model.load_state_dict(best_state)
    if hasattr(model, 'reset_position_state'):
        model.reset_position_state()
    print("Loaded best model state")

# Save checkpoint
ckpt_path = RESULTS_PATH / f"{prefix}_best_model.pth"
torch.save({
    'model_state_dict': model.state_dict(),
    'best': best,
    'metrics': {'val_sharpe': best_val_sharpe},
}, ckpt_path)
print(f"Saved checkpoint: {ckpt_path}")

## 5. Build OOS Dataset & Inference

In [ ]:
# ── Rebuild all samples for backtest (OOS = val split) ────────────────────────
all_samples = prep.build_samples(
    tech_lookback=tech_lookback,
    seq_lookback_bars=seq_lookback,
    numbars_lookback=numbars_lookback,
    use_fused_spatial=USE_FUSED_SPATIAL,
    stride=SAMPLE_STRIDE,
    sample_session=SAMPLE_SESSION,
    sample_session_start=SAMPLE_SESSION_START,
    sample_session_end=SAMPLE_SESSION_END,
)
print(f"Total samples (all): {len(all_samples):,}")

# ── Val split — last 25% by DATE ────────────────────────────────────────────
VAL_RATIO    = 0.25
all_dates    = [s['date'] for s in all_samples]
unique_dates = sorted(set(all_dates))
val_date_idx = int(len(unique_dates) * (1 - VAL_RATIO))
VAL_START_DATE = unique_dates[val_date_idx]
print(f"VAL_START_DATE    : {VAL_START_DATE}")

oos_samples = [s for s in all_samples if s['date'] >= VAL_START_DATE]
print(f"OOS samples       : {len(oos_samples):,}  ({VAL_START_DATE} -> {oos_samples[-1]['date']})")

sample_dates   = [s['date']   for s in oos_samples]
sample_tickers = [s['ticker'] for s in oos_samples]
for tk in TICKERS:
    n_tk = sum(1 for t in sample_tickers if t == tk)
    print(f"  {tk}: {n_tk:,} OOS samples")

dataset     = V3ContinuousDataset(
    oos_samples,
    fused_tail_shape=(FUSED_SPATIAL_CHANNELS, FUSED_SPATIAL_BINS),
)
full_loader = DataLoader(
    dataset, batch_size=BACKTEST_BATCH_SIZE,
    shuffle=False, collate_fn=v3_collate_fn, num_workers=0,
)
print(f"DataLoader  : {len(full_loader)} batches of up to {BACKTEST_BATCH_SIZE}")

## 6. Inference Pass

In [ ]:
bt_pos_list, bt_ret_list, bt_tid_list, bt_logits_list = [], [], [], []

model.eval()
if hasattr(model, 'reset_position_state'):
    model.reset_position_state()

with torch.no_grad():
    for batch in full_loader:
        inputs, targets = unpack_v3_batch(batch, device=device)
        out = model(**inputs, return_ae_losses=True)

        if USE_PTP:
            position, _ae_losses, logits = out
            bt_logits_list.append(logits.cpu())
        else:
            position, _ae_losses = out

        bt_pos_list.append(position.view(-1).cpu())
        bt_ret_list.append(targets.view(-1).float().cpu())
        bt_tid_list.append(inputs['ticker_id'].view(-1).cpu())

bt_pos    = torch.cat(bt_pos_list).numpy()
bt_ret    = torch.cat(bt_ret_list).numpy()
bt_tid    = torch.cat(bt_tid_list).numpy()
bt_logits = torch.cat(bt_logits_list).numpy() if bt_logits_list else None

id_to_ticker = {meta.ticker_id: t for t, meta in prep.registry.items()}
print(f"Inference done: {len(bt_pos):,} samples")
print(f"Position range: [{bt_pos.min():.4f}, {bt_pos.max():.4f}]  mean={bt_pos.mean():.4f}")
if bt_logits is not None:
    print(f"Logits shape  : {bt_logits.shape}")

## 7. Backtest DataFrame

In [ ]:
bt = pd.DataFrame({
    'date'        : sample_dates,
    'ticker'      : sample_tickers,
    'ticker_id'   : bt_tid.astype(int),
    'position'    : bt_pos,
    'fwd_return'  : bt_ret,
})

bt['date']         = pd.to_datetime(bt['date'])
bt['strategy_ret'] = bt['position'] * bt['fwd_return']
bt['active']       = bt['position'].abs() > TRADE_THRESH

# ── Transaction cost: charge TC_COST on every direction flip ─────────────────
bt = bt.sort_values(['ticker', 'date']).reset_index(drop=True)
signs           = np.sign(bt['position'].values)
ticker_boundary = (bt['ticker'] != bt['ticker'].shift(1)).values
sign_flip       = np.concatenate([[False], np.diff(signs) != 0])
sign_flip[ticker_boundary] = False

bt['tc_cost']         = np.where(sign_flip, TC_COST, 0.0)
bt['strategy_ret_tc'] = bt['strategy_ret'] - bt['tc_cost']
bt['is_trade']        = sign_flip

for col in ('strategy_ret', 'strategy_ret_tc'):
    bt[f'cum_{col}'] = bt.groupby('ticker')[col].cumsum()

print(f"OOS backtest: {len(bt):,} samples, {bt['date'].min().date()} -> {bt['date'].max().date()}")
print(bt.head(3).to_string())

## 8. Portfolio-Level Metrics

In [ ]:
def _metrics(df, label, col='strategy_ret'):
    sr     = df[col].values
    pos    = df['position'].values
    ret    = df['fwd_return'].values
    active = df['active'].values

    cum     = np.cumsum(sr)
    mean_sr = sr.mean()
    std_sr  = sr.std() + 1e-8
    neg     = sr[sr < 0]
    dside   = float(np.sqrt((neg**2).mean())) if len(neg) > 0 else 1e-8
    gp      = sr[sr > 0].sum()
    gl      = np.abs(sr[sr < 0]).sum() + 1e-9
    run_max = np.maximum.accumulate(cum)
    mdd     = float((run_max - cum).max())

    correct = ((pos > 0) & (ret > 0)) | ((pos < 0) & (ret < 0))
    dir_acc = (correct & active).sum() / max(active.sum(), 1)
    win_rate = ((sr[active] > 0).mean() * 100) if active.sum() > 0 else 0.0

    sharpe_ann  = (mean_sr / std_sr) * ANNUALIZATION
    sortino_ann = (mean_sr / (dside + 1e-8)) * ANNUALIZATION
    annual_ret  = mean_sr * BARS_PER_YEAR
    calmar      = annual_ret / (mdd + 1e-9)

    n_trades = int(df['is_trade'].sum())
    avg_trade = df.loc[df['is_trade'], col].mean() if n_trades > 0 else 0.0

    return {
        'Label'           : label,
        'N Samples'       : len(sr),
        'Net PnL'         : round(float(sr.sum()), 5),
        'Ann. Return'     : round(float(annual_ret), 5),
        'Sharpe (ann)'    : round(float(sharpe_ann), 3),
        'Sortino (ann)'   : round(float(sortino_ann), 3),
        'Calmar'          : round(float(calmar), 3),
        'Win Rate (%)'    : round(float(win_rate), 2),
        'Dir Acc (%)'     : round(float(dir_acc * 100), 2),
        'Profit Factor'   : round(float(gp / gl), 4),
        'Max Drawdown'    : round(mdd, 5),
        '# Trades'        : n_trades,
        'Avg Trade PnL'   : round(float(avg_trade), 6),
        'Avg |Position|'  : round(float(np.abs(pos).mean()), 4),
        'Active Bars (%)'  : round(float(active.mean() * 100), 2),
    }

rows = []
rows.append(_metrics(bt, 'ALL -- OOS (gross)'))
for tid in sorted(id_to_ticker):
    tk = id_to_ticker[tid]
    df_tk = bt[bt['ticker'] == tk]
    if len(df_tk) == 0: continue
    rows.append(_metrics(df_tk, f'{tk} -- OOS (gross)'))

rows.append({k: '------' if k != 'Label' else '---------------------' for k in rows[0]})

rows.append(_metrics(bt, 'ALL -- OOS (TC adj)', col='strategy_ret_tc'))
for tid in sorted(id_to_ticker):
    tk = id_to_ticker[tid]
    df_tk = bt[bt['ticker'] == tk]
    if len(df_tk) == 0: continue
    rows.append(_metrics(df_tk, f'{tk} -- OOS (TC adj)', col='strategy_ret_tc'))

df_summary = pd.DataFrame(rows).set_index('Label')
print("=" * 100)
print(f"OOS BACKTEST SUMMARY -- Trial 13 Transformer  (from {VAL_START_DATE})")
print("=" * 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print(df_summary.to_string())

## 9. Position Evolution Over Time

In [ ]:
daily_pos = (
    bt.groupby(['date', 'ticker'])['position']
    .mean()
    .unstack('ticker')
    .fillna(0)
)
daily_pos['COMBINED'] = daily_pos.mean(axis=1)
roll5 = daily_pos.rolling(5, min_periods=1).mean()

palette = {'CL': '#2980b9', 'GC': '#e67e22', 'COMBINED': 'black'}

fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

ax = axes[0]
for col in [c for c in daily_pos.columns if c != 'COMBINED']:
    ax.plot(daily_pos.index, daily_pos[col], alpha=0.5, lw=0.8,
            color=palette.get(col), label=col)
ax.axhline(0, color='gray', lw=0.8, linestyle=':')
ax.set_title('Daily Average Position by Ticker (OOS)')
ax.set_ylabel('Avg Position')
ax.legend()
ax.grid(True, alpha=0.25)

ax = axes[1]
for col in daily_pos.columns:
    ax.plot(roll5.index, roll5[col], lw=1.5,
            color=palette.get(col, 'gray'), label=col)
ax.axhline(0, color='gray', lw=0.8, linestyle=':')
ax.set_title('Rolling 5-Day Avg Position (OOS)')
ax.set_ylabel('Avg Position')
ax.legend(ncol=4)
ax.grid(True, alpha=0.25)

ax = axes[2]
hm_data = (
    bt.assign(ym=bt['date'].dt.to_period('M'))
    .groupby(['ym', 'ticker'])['position']
    .apply(lambda x: x.abs().mean())
    .unstack('ticker')
)
hm_data.index = hm_data.index.astype(str)
im = ax.imshow(hm_data.T.values, aspect='auto', cmap='RdYlGn',
               vmin=0, vmax=hm_data.values.max())
ax.set_xticks(range(len(hm_data.index)))
ax.set_xticklabels(hm_data.index, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(len(hm_data.columns)))
ax.set_yticklabels(hm_data.columns)
ax.set_title('Monthly Avg Absolute Position (Exposure) Heatmap')
plt.colorbar(im, ax=ax, fraction=0.015, pad=0.01)

plt.tight_layout()
plt.savefig(BACKTEST_PATH / f"{prefix}_bt_position_evolution.png", dpi=150, bbox_inches='tight')
plt.show()

## 10. Calibration — Position vs Realized Return Quantile

In [ ]:
N_QUANTILES = 10

fig, axes = plt.subplots(2, len(TICKERS) + 1, figsize=(7 * (len(TICKERS) + 1), 10))

for col_idx, (label, df_slice) in enumerate(
    [('ALL', bt)] + [(tk, bt[bt['ticker'] == tk]) for tk in sorted(id_to_ticker.values())]
):
    df_q = df_slice.copy()
    df_q['ret_q'] = pd.qcut(df_q['fwd_return'], q=N_QUANTILES, labels=False)

    agg = df_q.groupby('ret_q').agg(
        avg_position=('position', 'mean'),
        avg_strat_ret=('strategy_ret', 'mean'),
        avg_fwd_ret=('fwd_return', 'mean'),
        count=('position', 'count'),
    ).reset_index()

    q_labels = [f"D{i+1}" for i in range(N_QUANTILES)]
    cmap_vals = plt.cm.RdYlGn(np.linspace(0, 1, N_QUANTILES))

    ax = axes[0, col_idx]
    ax.bar(q_labels, agg['avg_position'].values, color=cmap_vals, alpha=0.9)
    ax.axhline(0, color='gray', lw=0.8, linestyle=':')
    ax.set_title(f'{label} -- Avg Position by Return Decile')
    ax.set_xlabel('Return Decile (D1=worst, D10=best)')
    ax.set_ylabel('Avg Position')
    ax.grid(True, alpha=0.25, axis='y')

    ax = axes[1, col_idx]
    ax.bar(q_labels, agg['avg_strat_ret'].values, color=cmap_vals, alpha=0.9)
    ax.axhline(0, color='gray', lw=0.8, linestyle=':')
    ax.set_title(f'{label} -- Avg Strategy Return by Decile')
    ax.set_xlabel('Return Decile')
    ax.set_ylabel('Avg Strategy Return')
    ax.grid(True, alpha=0.25, axis='y')

plt.suptitle('Position Calibration Across Realized Return Quantiles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f"{prefix}_bt_calibration.png", dpi=150, bbox_inches='tight')
plt.show()

## 11. Position-Return Scatter

In [ ]:
fig, axes = plt.subplots(1, len(TICKERS), figsize=(8 * len(TICKERS), 5), sharey=False)

for ax, tk in zip(axes if len(TICKERS) > 1 else [axes], sorted(id_to_ticker.values())):
    df_tk = bt[bt['ticker'] == tk].sample(n=min(5000, len(bt[bt['ticker'] == tk])),
                                           random_state=42)
    sc = ax.scatter(
        df_tk['position'], df_tk['fwd_return'],
        c=df_tk['strategy_ret'], cmap='RdYlGn',
        alpha=0.35, s=8, vmin=-0.003, vmax=0.003,
    )
    coefs = np.polyfit(df_tk['position'], df_tk['fwd_return'], 1)
    xs = np.linspace(df_tk['position'].min(), df_tk['position'].max(), 100)
    ax.plot(xs, np.polyval(coefs, xs), 'k--', lw=1.5, label=f"slope={coefs[0]:.4f}")
    ax.axhline(0, color='gray', lw=0.8, linestyle=':')
    ax.axvline(0, color='gray', lw=0.8, linestyle=':')
    ax.set_title(f'{tk} -- Position vs Realized Return')
    ax.set_xlabel('Position')
    ax.set_ylabel('Realized Forward Return')
    ax.legend(fontsize=9)
    plt.colorbar(sc, ax=ax, label='Strategy Ret')

plt.suptitle('Position-Return Scatter (colour = strategy return)', fontsize=13)
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f"{prefix}_bt_scatter.png", dpi=150, bbox_inches='tight')
plt.show()

## 12. Trade-Level Analysis

In [ ]:
def extract_trades(df_ticker):
    df = df_ticker.sort_values('date').reset_index(drop=True)
    pos = df['position'].values
    ret = df['strategy_ret_tc'].values
    active = np.abs(pos) > TRADE_THRESH

    trades = []
    in_trade = False
    t_start = None
    t_sign  = 0
    t_ret   = []

    for i in range(len(pos)):
        sign_i = int(np.sign(pos[i]))
        if active[i]:
            if not in_trade or sign_i != t_sign:
                if in_trade and t_ret:
                    trades.append({'sign': t_sign, 'n_bars': len(t_ret),
                                   'gross_ret': sum(t_ret)})
                in_trade = True
                t_sign   = sign_i
                t_ret    = [ret[i]]
            else:
                t_ret.append(ret[i])
        else:
            if in_trade and t_ret:
                trades.append({'sign': t_sign, 'n_bars': len(t_ret),
                               'gross_ret': sum(t_ret)})
            in_trade = False
            t_ret    = []
            t_sign   = 0

    if in_trade and t_ret:
        trades.append({'sign': t_sign, 'n_bars': len(t_ret), 'gross_ret': sum(t_ret)})

    return pd.DataFrame(trades) if trades else pd.DataFrame(
        columns=['sign', 'n_bars', 'gross_ret'])

trade_rows = []
n_tickers = len(TICKERS)
fig, axes = plt.subplots(2, max(n_tickers, 1), figsize=(8 * max(n_tickers, 1), 10),
                         squeeze=False)

for col_idx, tk in enumerate(sorted(id_to_ticker.values())):
    df_tk = bt[bt['ticker'] == tk]
    trades = extract_trades(df_tk)
    if len(trades) == 0:
        trade_rows.append({'Ticker': tk, '# Trades': 0})
        continue

    wins  = trades['gross_ret'] > 0
    longs = trades['sign']  == 1
    row = {
        'Ticker'           : tk,
        '# Trades'         : len(trades),
        '# Long'           : int(longs.sum()),
        '# Short'          : int((~longs).sum()),
        'Avg Profit/Trade' : round(trades['gross_ret'].mean(), 6),
        'Long Avg Profit'  : round(trades.loc[longs,  'gross_ret'].mean(), 6) if longs.any() else 0,
        'Short Avg Profit' : round(trades.loc[~longs, 'gross_ret'].mean(), 6) if (~longs).any() else 0,
        'Win Rate (%)'     : round(wins.mean() * 100, 2),
        'Long Win (%)'     : round(wins[longs].mean()  * 100, 2) if longs.any()  else 0,
        'Short Win (%)'    : round(wins[~longs].mean() * 100, 2) if (~longs).any() else 0,
        'Avg Hold (bars)'  : round(trades['n_bars'].mean(), 1),
        'Best Trade'       : round(trades['gross_ret'].max(), 6),
        'Worst Trade'      : round(trades['gross_ret'].min(), 6),
    }
    trade_rows.append(row)

    ax0 = axes[0, col_idx]
    ax1 = axes[1, col_idx]

    colors = ['#27ae60' if v > 0 else '#e74c3c' for v in trades['gross_ret']]
    ax0.bar(range(len(trades)), trades['gross_ret'].values,
            color=colors, alpha=0.75, width=1.0)
    ax0.axhline(0, color='gray', lw=0.8, linestyle=':')
    ax0.set_title(f'{tk} OOS -- Trade PnL (TC-adj)')
    ax0.set_xlabel('Trade #')
    ax0.set_ylabel('Gross Return')
    ax0.grid(True, alpha=0.25, axis='y')

    ax1.hist(trades['n_bars'].values, bins=30,
             color='steelblue', alpha=0.8, edgecolor='white')
    ax1.axvline(trades['n_bars'].mean(), color='red', lw=1.5,
                linestyle='--', label=f"mean={trades['n_bars'].mean():.1f}")
    ax1.set_title(f'{tk} OOS -- Holding Duration (bars)')
    ax1.set_xlabel('Bars Held')
    ax1.set_ylabel('Count')
    ax1.legend()
    ax1.grid(True, alpha=0.25)

df_trades = pd.DataFrame(trade_rows).set_index('Ticker')
print("\nTRADE-LEVEL ANALYSIS (OOS)")
print("=" * 90)
print(df_trades.to_string())

plt.suptitle('Trade-Level Analysis (OOS, TC-adjusted)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f"{prefix}_bt_trades.png", dpi=150, bbox_inches='tight')
plt.show()

## 13. Cumulative PnL & Rolling Sharpe

In [ ]:
ROLL_WINDOW = max(BARS_PER_DAY * 21, 200)

fig, axes = plt.subplots(3, 1, figsize=(16, 13), sharex=False)

# 1. Cumulative gross PnL
ax = axes[0]
palette_tk = plt.cm.tab10(np.linspace(0, 0.4, len(TICKERS)))
for tid, color in zip(sorted(id_to_ticker), palette_tk):
    tk   = id_to_ticker[tid]
    df_tk = bt[bt['ticker'] == tk].sort_values('date')
    cum   = df_tk['strategy_ret'].cumsum().values
    ax.plot(df_tk['date'].values, cum, lw=1.5, color=color, alpha=0.85, label=tk)

comb_daily = bt.groupby('date')['strategy_ret'].sum().sort_index().cumsum()
ax.plot(comb_daily.index, comb_daily.values, lw=2.2, color='black',
        linestyle='--', label='Combined')
ax.fill_between(comb_daily.index, comb_daily.values, 0,
                where=comb_daily.values >= 0, color='green', alpha=0.06)
ax.fill_between(comb_daily.index, comb_daily.values, 0,
                where=comb_daily.values < 0, color='red', alpha=0.06)
ax.axhline(0, color='gray', lw=0.8, linestyle=':')
ax.set_title('Cumulative Gross PnL (OOS)')
ax.set_ylabel('Cumulative PnL')
ax.legend(ncol=4)
ax.grid(True, alpha=0.25)

# 2. TC-adjusted cumulative PnL
ax = axes[1]
for tid, color in zip(sorted(id_to_ticker), palette_tk):
    tk   = id_to_ticker[tid]
    df_tk = bt[bt['ticker'] == tk].sort_values('date')
    cum   = df_tk['strategy_ret_tc'].cumsum().values
    ax.plot(df_tk['date'].values, cum, lw=1.5, color=color, alpha=0.85, label=f'{tk} (TC)')

comb_tc = bt.groupby('date')['strategy_ret_tc'].sum().sort_index().cumsum()
ax.plot(comb_tc.index, comb_tc.values, lw=2.2, color='black', linestyle='--',
        label='Combined (TC)')
ax.axhline(0, color='gray', lw=0.8, linestyle=':')
ax.set_title(f'Cumulative TC-Adjusted PnL  ({TC_COST_BPS} bps per leg)')
ax.set_ylabel('Cumulative PnL')
ax.legend(ncol=4)
ax.grid(True, alpha=0.25)

# 3. Rolling Sharpe
ax = axes[2]
comb_sr  = bt.sort_values('date').groupby('date')['strategy_ret'].sum()
roll_mu  = comb_sr.rolling(ROLL_WINDOW, min_periods=BARS_PER_DAY * 5).mean()
roll_std = comb_sr.rolling(ROLL_WINDOW, min_periods=BARS_PER_DAY * 5).std() + 1e-8
roll_sh  = (roll_mu / roll_std) * ANNUALIZATION

ax.plot(roll_sh.index, roll_sh.values, color='steelblue', lw=1.2, label='Rolling Sharpe')
ax.axhline(0, color='gray', lw=0.8, linestyle=':')
ax.fill_between(roll_sh.index, roll_sh.values, 0,
                where=roll_sh.values >= 0, color='steelblue', alpha=0.15)
ax.fill_between(roll_sh.index, roll_sh.values, 0,
                where=roll_sh.values < 0, color='red', alpha=0.15)
ax.set_title(f'Rolling Annualised Sharpe (window={ROLL_WINDOW} bars)')
ax.set_ylabel('Sharpe')
ax.legend()
ax.grid(True, alpha=0.25)

plt.suptitle('Trial 13 OOS Backtest -- Cumulative PnL & Rolling Sharpe',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f"{prefix}_bt_cum_pnl.png", dpi=150, bbox_inches='tight')
plt.show()

## 14. Drawdown Analysis

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=False)

def _drawdown_series(cum_series):
    run_max = cum_series.cummax()
    return run_max - cum_series

ax = axes[0]
comb_cum = comb_tc
dd = _drawdown_series(comb_cum)
ax.fill_between(dd.index, dd.values, color='#e74c3c', alpha=0.5)
ax.plot(dd.index, dd.values, color='#c0392b', lw=0.8)
ax.set_title(f'Combined Drawdown (TC-adj, OOS)   Max DD = {dd.max():.5f}')
ax.set_ylabel('Drawdown')
ax.grid(True, alpha=0.25)

ax = axes[1]
for tid, color in zip(sorted(id_to_ticker), palette_tk):
    tk    = id_to_ticker[tid]
    df_tk = bt[bt['ticker'] == tk].sort_values('date')
    cum_t = df_tk.set_index('date')['strategy_ret_tc'].cumsum()
    dd_t  = _drawdown_series(cum_t)
    ax.plot(dd_t.index, dd_t.values, color=color, lw=1.3, alpha=0.85,
            label=f'{tk} (max={dd_t.max():.5f})')
ax.set_title('Per-Ticker Drawdown (TC-adj, OOS)')
ax.set_ylabel('Drawdown')
ax.legend()
ax.grid(True, alpha=0.25)

plt.suptitle('Drawdown Analysis -- Trial 13 (OOS)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f"{prefix}_bt_drawdown.png", dpi=150, bbox_inches='tight')
plt.show()

## 15. Monthly PnL Calendar Heatmap

In [ ]:
fig, axes = plt.subplots(1, len(TICKERS) + 1, figsize=(7 * (len(TICKERS) + 1), 5))

slices = [('ALL', bt)] + [(tk, bt[bt['ticker'] == tk])
                           for tk in sorted(id_to_ticker.values())]

for ax, (label, df_s) in zip(axes, slices):
    df_s = df_s.copy()
    df_s['year']  = df_s['date'].dt.year
    df_s['month'] = df_s['date'].dt.month

    pivot = (
        df_s.groupby(['year', 'month'])['strategy_ret_tc']
        .sum()
        .unstack('month')
    )
    pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                     'Jul','Aug','Sep','Oct','Nov','Dec'][:len(pivot.columns)]
    pivot = pivot.reindex(columns=['Jan','Feb','Mar','Apr','May','Jun',
                                   'Jul','Aug','Sep','Oct','Nov','Dec'])

    abs_max = pivot.abs().max().max()
    sns.heatmap(
        pivot, ax=ax, cmap='RdYlGn',
        center=0, vmin=-abs_max, vmax=abs_max,
        annot=True, fmt='.4f', annot_kws={'size': 7},
        linewidths=0.5, cbar_kws={'shrink': 0.8},
    )
    ax.set_title(f'{label} -- Monthly PnL (TC-adj)')
    ax.set_xlabel('')

plt.suptitle('Monthly PnL Calendar Heatmap -- Trial 13', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f"{prefix}_bt_monthly_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

## 16. Quantile Head — Class Logit Distribution

In [ ]:
if USE_PTP and bt_logits is not None:
    import torch.nn.functional as F
    from CTAFlow.models.deep_learning.multi_branch.tft import PredictionToPosition

    N_CLS    = PredictionToPosition.N_CLASSES
    probs    = torch.softmax(torch.tensor(bt_logits), dim=-1).numpy()
    pred_cls = probs.argmax(axis=1)

    outer_threshold = float(best.get('outer_threshold', 1.0))
    true_cls = returns_to_classes(
        torch.tensor(bt_ret), outer=outer_threshold,
    ).numpy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    class_labels = ['Q1\n(Strong -)', 'Q2\n(Weak -)', 'Q3\n(Weak +)', 'Q4\n(Strong +)']
    colors_cls   = plt.cm.RdYlGn(np.linspace(0, 1, N_CLS))
    for c in range(N_CLS):
        ax.hist(probs[:, c], bins=50, alpha=0.55, color=colors_cls[c],
                label=class_labels[c], density=True)
    ax.set_title('Class Probability Distributions (OOS)')
    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

    ax = axes[1]
    conf = np.zeros((N_CLS, N_CLS), dtype=int)
    for t, p in zip(true_cls, pred_cls):
        conf[int(t), int(p)] += 1
    conf_row_sums = conf.sum(axis=1, keepdims=True)
    conf_norm = np.divide(conf, conf_row_sums, out=np.zeros_like(conf, dtype=float), where=conf_row_sums > 0)
    sns.heatmap(conf_norm, ax=ax, cmap='Blues', annot=True, fmt='.2f',
                xticklabels=class_labels, yticklabels=class_labels,
                vmin=0, vmax=1)
    ax.set_title('Normalised Confusion Matrix (OOS)')
    ax.set_xlabel('Predicted Class')
    ax.set_ylabel('True Class')

    plt.suptitle('QuantileHead -- Class Logit Analysis (4 classes)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(BACKTEST_PATH / f"{prefix}_bt_class_logits.png", dpi=150, bbox_inches='tight')
    plt.show()

    print("\nOOS Classification Accuracy by Class:")
    for c in range(N_CLS):
        mask_c = true_cls == c
        acc_c  = (pred_cls[mask_c] == c).mean() * 100 if mask_c.sum() > 0 else 0
        print(f"  {class_labels[c].replace(chr(10), ' ')}: {acc_c:.1f}%  (n={mask_c.sum()})")
    overall_acc = (pred_cls == true_cls).mean() * 100
    print(f"  Overall: {overall_acc:.2f}%")
else:
    print("USE_PTP=False or logits not collected -- skipping class analysis.")

## 17. Branch Attribution & Leakage Detection

In [ ]:
# Resolve sequential VPIN column names
_seq_cols = None
for tk in TICKERS:
    if tk in prep._seq_vpin and len(prep._seq_vpin[tk].columns) > 0:
        _seq_cols = list(prep._seq_vpin[tk].columns)
        break
if _seq_cols is None:
    _seq_cols = [f"seq_{i}" for i in range(F_SEQ)]
tech_cols = list(prep._tech_feature_cols)

print(f"Sequential VPIN features ({len(_seq_cols)}): {_seq_cols}")
print(f"Tech features ({len(tech_cols)}): {tech_cols[:10]}{'...' if len(tech_cols) > 10 else ''}")

# Hook to capture branch weights
_captured_bw = []

def _bvs_hook(module, inputs, outputs):
    _z_selected, weights = outputs
    _captured_bw.append(weights.detach().cpu().numpy())

hook_handle = model.base_model.branch_selector.register_forward_hook(_bvs_hook)

temporal_attn_all = []
attr_positions = []
attr_returns = []
attr_tickers = []

model.eval()
if hasattr(model, 'reset_position_state'):
    model.reset_position_state()

with torch.no_grad():
    for batch in full_loader:
        inputs, targets = unpack_v3_batch(batch, device=device)
        out = model(**inputs, return_ae_losses=True)

        if USE_PTP:
            position, _ae_losses, _logits = out
        else:
            position, _ae_losses = out

        tracker = model.base_model._last_tracker
        ta = tracker.get("temporal_attn")
        if ta is not None:
            if isinstance(ta, torch.Tensor):
                ta = ta.cpu().numpy()
            temporal_attn_all.append(ta)

        attr_positions.append(position.view(-1).cpu().numpy())
        attr_returns.append(targets.view(-1).float().cpu().numpy())
        attr_tickers.append(inputs['ticker_id'].view(-1).cpu().numpy())

hook_handle.remove()

branch_weights_arr = np.concatenate(_captured_bw, axis=0)
attr_pos_arr = np.concatenate(attr_positions)
attr_ret_arr = np.concatenate(attr_returns)
attr_tid_arr = np.concatenate(attr_tickers)

print(f"\nCollected attribution for {len(attr_pos_arr):,} OOS samples")
print(f"Branch weights shape: {branch_weights_arr.shape}")

# Static sequential feature gate weights
seq_net = model.base_model.seq_net
seq_feature_weights = torch.softmax(seq_net.feature_gate_logits, dim=-1).detach().cpu().numpy()
print(f"\nSequential feature gate weights ({len(seq_feature_weights)} features):")
for name, w in sorted(zip(_seq_cols, seq_feature_weights), key=lambda x: -x[1]):
    bar = chr(9608) * int(w * 200)
    print(f"  {name:25s}: {w:.4f}  {bar}")

In [ ]:
from scipy import stats

branch_names = ['Backbone (Tech)', 'Spatial (NB+VPIN)', 'Sequential (VPIN)']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Overall branch weight distribution
ax = axes[0, 0]
bw_means = branch_weights_arr.mean(axis=0)
bw_stds = branch_weights_arr.std(axis=0)
colors_bw = ['#3498db', '#e67e22', '#2ecc71']
bars = ax.bar(branch_names, bw_means, yerr=bw_stds, color=colors_bw,
              alpha=0.85, capsize=5, edgecolor='white', linewidth=1.5)
for bar, m in zip(bars, bw_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{m:.3f}', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Mean Branch Weights (OOS)', fontsize=13)
ax.set_ylabel('Weight (softmax)')
ax.set_ylim(0, min(1.0, bw_means.max() + bw_stds.max() + 0.05))
ax.grid(True, alpha=0.25, axis='y')

# 2. Sequential feature gate importance
ax = axes[0, 1]
sort_idx = np.argsort(seq_feature_weights)[::-1]
sorted_names = [_seq_cols[i] for i in sort_idx]
sorted_weights = seq_feature_weights[sort_idx]
n_show = min(20, len(sorted_names))
ax.barh(range(n_show), sorted_weights[:n_show],
        color='#2ecc71', alpha=0.85, edgecolor='white')
ax.set_yticks(range(n_show))
ax.set_yticklabels(sorted_names[:n_show], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Gate Weight (softmax)')
ax.set_title('Sequential Branch -- Feature Gate Importance', fontsize=13)
ax.grid(True, alpha=0.25, axis='x')

# 3. Per-ticker branch weights
ax = axes[1, 0]
x_pos = np.arange(len(branch_names))
width = 0.8 / max(len(TICKERS), 1)
for j, tid in enumerate(sorted(id_to_ticker)):
    tk = id_to_ticker[tid]
    mask = attr_tid_arr == tid
    tk_bw = branch_weights_arr[mask].mean(axis=0)
    offset = (j - len(TICKERS)/2 + 0.5) * width
    ax.bar(x_pos + offset, tk_bw, width=width, label=tk, alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels(branch_names)
ax.set_ylabel('Mean Weight')
ax.set_title('Branch Weights by Ticker (OOS)', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.25, axis='y')

# 4. LEAKAGE TEST
ax = axes[1, 1]
leak_results = []
for i, bname in enumerate(branch_names):
    bw_col = branch_weights_arr[:, i]
    corr, pval = stats.pearsonr(bw_col, attr_ret_arr)
    leak_results.append((bname, corr, pval))
    rho, rho_p = stats.spearmanr(bw_col, attr_ret_arr)
    leak_results.append((f'{bname} (rank)', rho, rho_p))

leak_df = pd.DataFrame(leak_results, columns=['Branch', 'Correlation', 'p-value'])
colors_leak = ['#e74c3c' if abs(r) > 0.05 and p < 0.01 else '#27ae60'
               for r, p in zip(leak_df['Correlation'], leak_df['p-value'])]
ax.barh(range(len(leak_df)), leak_df['Correlation'].values,
        color=colors_leak, alpha=0.85, edgecolor='white')
ax.set_yticks(range(len(leak_df)))
ax.set_yticklabels(leak_df['Branch'].values, fontsize=9)
ax.axvline(0, color='gray', lw=0.8, linestyle=':')
ax.axvline(0.05, color='red', lw=0.8, linestyle='--', alpha=0.5)
ax.axvline(-0.05, color='red', lw=0.8, linestyle='--', alpha=0.5)
ax.set_xlabel('Pearson / Spearman Correlation with Future Return')
ax.set_title('Leakage Test: Branch Weight vs Fwd Return (OOS)', fontsize=13)
ax.grid(True, alpha=0.25, axis='x')

plt.suptitle('Trial 13 Branch Attribution & Leakage Detection (OOS)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f"{prefix}_bt_attribution.png", dpi=150, bbox_inches='tight')
plt.show()

# Print leakage summary
print("\n" + "=" * 80)
print("LEAKAGE TEST SUMMARY")
print("=" * 80)
print(f"{'Branch':<30s} {'Corr':>8s} {'p-value':>10s} {'Flag':>6s}")
print("-" * 60)
for _, row in leak_df.iterrows():
    flag = "LEAK?" if abs(row['Correlation']) > 0.05 and row['p-value'] < 0.01 else "OK"
    print(f"  {row['Branch']:<28s} {row['Correlation']:>8.4f} {row['p-value']:>10.2e} {flag:>6s}")

pos_corr, pos_p = stats.pearsonr(attr_pos_arr, attr_ret_arr)
print(f"\nPosition vs Fwd Return: r={pos_corr:.4f}  p={pos_p:.2e}")
if pos_corr < 0:
    print("  WARNING: Negative position-return correlation -- model may be inverted!")
elif pos_corr > 0.15:
    print("  WARNING: Very high position-return correlation -- possible lookahead!")
else:
    print("  OK: Moderate positive correlation as expected.")

## 18. Temporal Attention & Sequential Feature Leakage Check

In [ ]:
if temporal_attn_all:
    attn_arr = np.concatenate(temporal_attn_all, axis=0)
    mean_attn = attn_arr.mean(axis=0)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    ax = axes[0]
    L = len(mean_attn)
    ax.bar(range(L), mean_attn, color='steelblue', alpha=0.8, width=1.0)
    ax.set_xlabel(f'Tech Lookback Position (0=oldest, {L-1}=most recent)')
    ax.set_ylabel('Mean Attention Weight')
    ax.set_title(f'Temporal Attention Distribution (avg over {len(attn_arr):,} OOS samples)')
    ax.grid(True, alpha=0.25, axis='y')

    ax = axes[1]
    for tid in sorted(id_to_ticker):
        tk = id_to_ticker[tid]
        mask = attr_tid_arr == tid
        tk_attn = attn_arr[mask].mean(axis=0)
        ax.plot(range(L), tk_attn, lw=1.5, label=tk, alpha=0.85)
    ax.set_xlabel('Tech Lookback Position')
    ax.set_ylabel('Mean Attention Weight')
    ax.set_title('Temporal Attention by Ticker (OOS)')
    ax.legend()
    ax.grid(True, alpha=0.25)

    plt.suptitle('Temporal Backbone Attention -- Which Bars Matter?',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(BACKTEST_PATH / f"{prefix}_bt_temporal_attn.png", dpi=150, bbox_inches='tight')
    plt.show()

    last_3_attn = attn_arr[:, -3:].mean(axis=1)
    corr_recent, p_recent = stats.pearsonr(last_3_attn, attr_ret_arr)
    print(f"Attention on last 3 bars vs fwd return: r={corr_recent:.4f}  p={p_recent:.2e}")
    if abs(corr_recent) > 0.05 and p_recent < 0.01:
        print("  WARNING: Recent-bar attention correlates with future return -- check for leakage!")
    else:
        print("  OK: No significant correlation between attention recency and future return.")
else:
    print("No temporal attention data collected.")

# Sequential Feature Leakage: per-feature correlation with future return
print("\n" + "=" * 80)
print("SEQUENTIAL FEATURE vs FWD RETURN CORRELATION (per-feature leakage check)")
print("=" * 80)
print("  Checking raw input features from the LAST batch for directional leakage...")

model.eval()
if hasattr(model, 'reset_position_state'):
    model.reset_position_state()

with torch.no_grad():
    for batch in full_loader:
        inputs, targets = unpack_v3_batch(batch, device=device)
        seq_raw = inputs['seq_vpin'].cpu().numpy()
        seq_lens = inputs['seq_vpin_lens'].cpu().numpy()
        fwd_ret = targets.view(-1).float().cpu().numpy()
        break

n_samples, T, n_feat = seq_raw.shape
feat_means = np.zeros((n_samples, n_feat))
for i in range(n_samples):
    valid_len = int(seq_lens[i])
    if valid_len > 0:
        feat_means[i] = seq_raw[i, :valid_len, :].mean(axis=0)

print(f"\n{'Feature':<25s} {'Pearson r':>10s} {'p-value':>10s} {'Gate Wt':>10s} {'Flag':>8s}")
print("-" * 70)
suspicious = []
for j in range(n_feat):
    fname = _seq_cols[j] if j < len(_seq_cols) else f"seq_{j}"
    r, p = stats.pearsonr(feat_means[:, j], fwd_ret)
    gw = seq_feature_weights[j] if j < len(seq_feature_weights) else 0
    flag = "SUSPECT" if abs(r) > 0.10 and p < 0.01 else ""
    if flag:
        suspicious.append(fname)
    print(f"  {fname:<23s} {r:>10.4f} {p:>10.2e} {gw:>10.4f} {flag:>8s}")

if suspicious:
    print(f"\n  SUSPICIOUS features (|r|>0.10, p<0.01): {suspicious}")
    print("  -> These features may contain forward-looking information!")
else:
    print("\n  No features show suspicious correlation with future returns.")

## 19. Export Results

In [ ]:
bt_export_path = BACKTEST_PATH / f"{prefix}_bt_full.csv"
bt.to_csv(bt_export_path, index=False)
print(f"Full backtest saved -> {bt_export_path}")

summary_path = BACKTEST_PATH / f"{prefix}_bt_summary.csv"
df_summary.to_csv(summary_path)
print(f"Summary saved       -> {summary_path}")

trades_path = BACKTEST_PATH / f"{prefix}_bt_trades.csv"
df_trades.to_csv(trades_path)
print(f"Trade table saved   -> {trades_path}")

print("\n" + "=" * 70)
print(f"BACKTEST COMPLETE -- OOS from {VAL_START_DATE}")
print("=" * 70)
for col_label, col in [('Gross', 'strategy_ret'), ('TC-adj', 'strategy_ret_tc')]:
    sr = bt[col].values
    ann_sh = (sr.mean() / (sr.std() + 1e-8)) * ANNUALIZATION
    print(f"  {col_label:8s}: Ann.Sharpe={ann_sh:.3f}   "
          f"Net PnL={sr.sum():.5f}   "
          f"Max DD={_metrics(bt, '', col=col)['Max Drawdown']:.5f}")